# UK Accident Data Cleaning

This notebook performs clean, practical preprocessing for `UK_Accident.csv` using only core pandas steps.

Cleaning goals:
- Remove index-like junk columns and duplicates
- Standardize column names and text values
- Convert date/time and numeric fields to proper dtypes
- Handle obvious invalid ranges (lat/long, speed, casualty counts)
- Export cleaned datasets for analysis and modeling

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 60)

DATA_PATH = Path('UK_Accident.csv')
CLEAN_PATH = Path('UK_Accident_cleaned.csv')
MODEL_READY_PATH = Path('UK_Accident_model_ready.csv')

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f'Raw shape: {df_raw.shape}')
df_raw.head()

## 1) Raw Data Check
Quickly inspect types, missingness, and duplicates before cleaning.

In [ ]:
print(df_raw.info())

missing_pct = (df_raw.isna().mean() * 100).sort_values(ascending=False)
print('\nTop missing % columns:')
print(missing_pct.head(10))

dup_all = df_raw.duplicated().sum()
dup_index = df_raw.duplicated(subset=['Accident_Index']).sum() if 'Accident_Index' in df_raw.columns else 0
print(f'\nDuplicate full rows: {dup_all}')
print(f'Duplicate Accident_Index values: {dup_index}')

## 2) Cleaning Function
The function below applies consistent, explainable cleaning steps.

In [ ]:
def clean_uk_accident_data(df: pd.DataFrame) -> pd.DataFrame:
    clean = df.copy()

    # 1) Remove auto-generated unnamed columns (often saved index columns)
    unnamed_cols = [c for c in clean.columns if str(c).strip().lower().startswith('unnamed')]
    if unnamed_cols:
        clean = clean.drop(columns=unnamed_cols)

    # 2) Standardize column names to snake_case
    clean.columns = (
        clean.columns
        .str.strip()
        .str.lower()
        .str.replace(r'[^a-z0-9]+', '_', regex=True)
        .str.strip('_')
    )

    # 3) Trim text and convert empty strings to NaN
    object_cols = clean.select_dtypes(include='object').columns
    for col in object_cols:
        clean[col] = clean[col].astype(str).str.strip()
        clean[col] = clean[col].replace({'': np.nan, 'nan': np.nan, 'NaN': np.nan})

    # 4) Replace common placeholder strings with NaN
    placeholder_tokens = [
        'Data missing or out of range',
        'Unknown',
        'Not known',
        'Not available'
    ]
    for col in clean.select_dtypes(include='object').columns:
        clean[col] = clean[col].replace(placeholder_tokens, np.nan)

    # 5) Parse date and time
    if 'date' in clean.columns:
        clean['date'] = pd.to_datetime(clean['date'], dayfirst=True, errors='coerce')

    if 'time' in clean.columns:
        clean['time'] = pd.to_datetime(clean['time'], format='%H:%M', errors='coerce').dt.time

    # 6) Convert known numeric columns
    numeric_candidates = [
        'location_easting_osgr', 'location_northing_osgr',
        'longitude', 'latitude',
        'police_force', 'accident_severity',
        'number_of_vehicles', 'number_of_casualties',
        'day_of_week', 'local_authority_district',
        '1st_road_class', '1st_road_number',
        'speed_limit', '2nd_road_class', '2nd_road_number',
        'urban_or_rural_area',
        'year'
    ]

    for col in numeric_candidates:
        if col in clean.columns:
            clean[col] = pd.to_numeric(clean[col], errors='coerce')

    # 7) Basic range validation
    if 'latitude' in clean.columns:
        clean.loc[~clean['latitude'].between(49, 61), 'latitude'] = np.nan

    if 'longitude' in clean.columns:
        clean.loc[~clean['longitude'].between(-9, 2), 'longitude'] = np.nan

    if 'speed_limit' in clean.columns:
        clean.loc[~clean['speed_limit'].between(10, 80), 'speed_limit'] = np.nan

    for col in ['number_of_vehicles', 'number_of_casualties']:
        if col in clean.columns:
            clean.loc[clean[col] < 0, col] = np.nan

    # 8) Remove duplicates
    clean = clean.drop_duplicates()
    if 'accident_index' in clean.columns:
        clean = clean.drop_duplicates(subset=['accident_index'], keep='first')

    return clean

In [ ]:
df_clean = clean_uk_accident_data(df_raw)

print(f'Clean shape: {df_clean.shape}')
print(f'Columns after cleaning: {len(df_clean.columns)}')

clean_missing_pct = (df_clean.isna().mean() * 100).sort_values(ascending=False)
print('\nTop missing % columns after cleaning:')
print(clean_missing_pct.head(10))

df_clean.head()

## 3) Optional Model-Ready Version
This keeps your cleaned data and also creates an imputed version if you need a fully non-null table for ML.

In [ ]:
df_model = df_clean.copy()

num_cols = df_model.select_dtypes(include=['number']).columns
cat_cols = df_model.select_dtypes(include=['object']).columns

df_model[num_cols] = df_model[num_cols].fillna(df_model[num_cols].median())

for col in cat_cols:
    mode_vals = df_model[col].mode(dropna=True)
    fill_val = mode_vals.iloc[0] if len(mode_vals) > 0 else 'Unknown'
    df_model[col] = df_model[col].fillna(fill_val)

print('Model-ready missing values:', int(df_model.isna().sum().sum()))

In [ ]:
df_clean.to_csv(CLEAN_PATH, index=False)
df_model.to_csv(MODEL_READY_PATH, index=False)

print(f'Saved cleaned dataset to: {CLEAN_PATH.resolve()}')
print(f'Saved model-ready dataset to: {MODEL_READY_PATH.resolve()}')